# VL-JEPA 2 — Vision-Language JEPA · SFT Training

Runs the **supervised fine-tuning (SFT)** stage for VL-JEPA 2, on top of an
(optional) pretrained checkpoint. The `Trainer` handles gradient accumulation,
gradient clipping, LR scheduling, mixed precision, and periodic checkpointing.

### What this notebook does
1. Load an SFT config (`sft_tiny.yaml` for a smoke test, or `sft.yaml`).
2. Validate config, set seeds, and resolve the device.
3. Build the model, data loader, optimizer, and scheduler.
4. Run `Trainer.fit` — checkpoints go to `cfg["train"]["output_dir"]`, which the
   inference notebook can then load.


In [1]:
# Enables postponed annotation evaluation (safe forward references).
from __future__ import annotations

In [2]:
# Training building blocks + config/seed utilities from the vl_jepa2 package.
import torch

from vl_jepa2.train.build import build_model, build_optimizer_and_scheduler, build_train_loader
from vl_jepa2.train.trainer import Trainer
from vl_jepa2.utils.config import dump_resolved_config, load_yaml, validate_train_config
from vl_jepa2.utils.seed import set_seed

In [3]:
# Reference CLI (kept as documentation of the original training script).
"""
parser = argparse.ArgumentParser(description="Train VL-JEPA SFT stage")
parser.add_argument("--config", type=str, default="vljepa/configs/sft.yaml")
parser.add_argument("--checkpoint", type=str, default=None, help="Optional base checkpoint from pretraining")
args = parser.parse_args()
"""

'\nparser = argparse.ArgumentParser(description="Train VL-JEPA SFT stage")\nparser.add_argument("--config", type=str, default="vljepa/configs/sft.yaml")\nparser.add_argument("--checkpoint", type=str, default=None, help="Optional base checkpoint from pretraining")\nargs = parser.parse_args()\n'

In [4]:
# Use sft_tiny.yaml for a quick smoke test; switch to sft.yaml for a real run.
#cfg = load_yaml("./vl_jepa2/configs/sft.yaml")
cfg = load_yaml("./vl_jepa2/configs/sft_tiny.yaml")

In [5]:
# Validate the config, then seed everything for deterministic training.
validate_train_config(cfg)
set_seed(cfg["train"]["seed"], deterministic=True)

In [6]:
# Persist the fully-resolved config alongside checkpoints for reproducibility.
dump_resolved_config(cfg, cfg["train"]["output_dir"])
device = torch.device(cfg["train"]["device"] if torch.cuda.is_available() else "cpu")

In [7]:
model = build_model(cfg)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: Qwen/Qwen3-1.7B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/sagemaker-user/rlvr_lwm/vl_jepa2/models/vljepa.py:247: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.predictor = nn.TransformerEncoder(encoder_layer, num_layers=cfg.predictor_layers)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [8]:
# Optional: warm-start from a pretraining checkpoint before SFT.
"""
# if checkpoint exists (i.e. training on top)
ckpt = torch.load(args.checkpoint, map_location="cpu")
model.load_state_dict(ckpt["model"], strict=False)
"""

'\n# if checkpoint exists (i.e. training on top)\nckpt = torch.load(args.checkpoint, map_location="cpu")\nmodel.load_state_dict(ckpt["model"], strict=False)\n'

In [9]:
# Build the data loader plus optimizer & LR scheduler from config.
loader = build_train_loader(cfg)
optimizer, scheduler = build_optimizer_and_scheduler(model, cfg)

In [10]:
# grad_accum_steps simulates a larger batch; temperature scales the
# contrastive-style objective; precision controls AMP dtype.
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=loader,
    output_dir=cfg["train"]["output_dir"],
    max_steps=cfg["train"]["max_steps"],
    grad_accum_steps=cfg["train"]["gradient_accumulation_steps"],
    clip_grad_norm=cfg["train"]["clip_grad_norm"],
    log_every=cfg["train"]["log_every"],
    save_every=cfg["train"]["save_every"],
    temperature=cfg["train"]["temperature"],
    precision=cfg["train"]["precision"],
)

In [11]:
# Run SFT. Checkpoints are written to cfg['train']['output_dir'].
trainer.fit(device)

training:   0%|          | 0/100 [00:00<?, ?it/s]/home/sagemaker-user/.conda/envs/rlwm2/lib/python3.12/site-packages/torch/autograd/graph.py:865: UserWarning: Flash Attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:114.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/home/sagemaker-user/.conda/envs/rlwm2/lib/python3.12/site-packages/torch/autograd/graph.py:865: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:897.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pas